# agentic_alpha_budget_overlay_search_v1

Diagnostic R-G-E-V-D card for CPPI/TIPP, volatility targeting, and trend-gated risk-budget overlays on the existing production qlib signal. The card reads production recorder predictions and labels, retrieves qlib SOXL/SOXX/VIX state through `D.features`, and validates whether any overlay can beat the current production strategy while holding MaxDD below 20%.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_budget_overlay_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
START = '2019-01-02'
END = '2026-06-09'
MAX_DRAWDOWN_FLOOR = -20.0
TRANSACTION_COST = 0.0003

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  test:
    start: {START}
    end: {END}
rgevd:
  branch: production_signal_budget_overlay
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')
production_summary = json.loads((PRODUCTION_CARD / 'summary.json').read_text(encoding='utf-8'))
production_recorder = Path(production_summary['recorder_local_dir'])

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Close each generated branch with evidence: high return alone is insufficient when validation rejects drawdown.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'arxiv',
            'title': 'Compounding Effects in Leveraged ETFs',
            'url': 'https://arxiv.org/html/2504.20116v1',
            'extracted_signal': 'Leveraged ETF compounding benefits from persistent trends but is harmed by volatility clustering.',
            'retrieval_status': 'prior_retrieval_seed',
        },
        {
            'source_type': 'paper',
            'title': 'Conditional volatility targeting',
            'url': 'https://alphaarchitect.com/conditional-volatility-targeting/',
            'extracted_signal': 'Reduce exposure in extreme volatility states and retain exposure in lower-risk states.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'CPPI / TIPP portfolio insurance overlays',
            'url': 'https://quantpedia.com/introduction-to-cppi-constant-proportion-portfolio-insurance/',
            'extracted_signal': 'Use a floor, cushion, multiplier, and cap to preserve upside while constraining drawdown budget.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'twitter',
            'title': 'SOXL volatility and drawdown risk discussion',
            'url': 'https://x.com/Alpha_Ex_LLC/status/2067644619462938886',
            'extracted_signal': 'SOXL risk controls need to react to volatility and drawdown state, not only raw signal strength.',
            'retrieval_status': 'search_snippet_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
pred = pd.read_pickle(production_recorder / 'artifacts' / 'pred.pkl').reset_index().rename(columns={'score': 'score'})
label = pd.read_pickle(production_recorder / 'artifacts' / 'label.pkl').reset_index()
label_col = [col for col in label.columns if col not in {'datetime', 'instrument'}][0]
label = label.rename(columns={label_col: 'fwd_ret'})
features = D.features(
    ['SOXL', 'SOXX'],
    ['$close', '$vix'],
    start_time='2010-03-11',
    end_time='2026-06-10',
    freq='day',
).reset_index()
soxl = features[features['instrument'] == 'SOXL'][['datetime', '$close', '$vix']].rename(columns={'$close': 'soxl_close', '$vix': 'vix'})
soxx = features[features['instrument'] == 'SOXX'][['datetime', '$close']].rename(columns={'$close': 'soxx_close'})
frame = pred.merge(label, on=['datetime', 'instrument']).merge(soxl, on='datetime').merge(soxx, on='datetime').sort_values('datetime').reset_index(drop=True)
frame['soxl_ret1'] = frame['soxl_close'].pct_change()
frame['soxx_ret1'] = frame['soxx_close'].pct_change()
for n in [20, 63, 126, 200]:
    frame[f'soxx_sma{n}'] = frame['soxx_close'].rolling(n).mean().shift(1)
    frame[f'soxx_ret{n}'] = frame['soxx_close'].pct_change(n).shift(1)
    frame[f'soxl_vol{n}'] = frame['soxl_ret1'].rolling(n).std().shift(1) * np.sqrt(252)
frame['vix_lag'] = frame['vix'].shift(1).fillna(frame['vix'])
frame = frame.loc[(frame['datetime'] >= pd.Timestamp(START)) & (frame['datetime'] <= pd.Timestamp(END))].reset_index(drop=True)

ret = frame['fwd_ret'].fillna(0.0).to_numpy(float)
score = frame['score'].to_numpy(float)
vix = frame['vix_lag'].to_numpy(float)
trend_flags = {
    'none': np.ones(len(frame), dtype=bool),
    'ret20_63': ((frame['soxx_ret20'] > 0) & (frame['soxx_ret63'] > 0)).fillna(False).to_numpy(bool),
    'ret63_pos': (frame['soxx_ret63'] > 0).fillna(False).to_numpy(bool),
    'sma200': (frame['soxx_close'].shift(1) > frame['soxx_sma200']).fillna(False).to_numpy(bool),
}
vol_inputs = {
    'none': None,
    'vol20': frame['soxl_vol20'].to_numpy(float),
    'vol63': frame['soxl_vol63'].to_numpy(float),
}


def path_metrics(weight: np.ndarray) -> dict[str, float]:
    w = np.nan_to_num(np.asarray(weight, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    w = np.clip(w, 0.0, 10.0)
    turnover = np.abs(np.diff(np.r_[0.0, w]))
    strategy_return = w * ret - turnover * TRANSACTION_COST
    nav = pd.Series(np.cumprod(1.0 + strategy_return), index=frame['datetime'])
    drawdown = nav / nav.cummax() - 1.0
    return {
        'return_pct': float((nav.iloc[-1] - 1.0) * 100.0),
        'max_drawdown_pct': float(drawdown.min() * 100.0),
        'turnover': float(turnover.sum()),
        'mean_exposure_pct': float(w.mean() * 100.0),
    }


def tipp_weight(raw: np.ndarray, *, floor_dd: float, multiplier: float, max_budget: float) -> np.ndarray:
    nav = 1.0
    high_watermark = 1.0
    weights: list[float] = []
    previous = 0.0
    for raw_weight, forward_return in zip(raw, ret, strict=True):
        high_watermark = max(high_watermark, nav)
        floor = high_watermark * (1.0 - floor_dd)
        cushion = max(0.0, (nav - floor) / nav) if nav > 0 else 0.0
        budget = min(max_budget, multiplier * cushion)
        weight = float(min(max(raw_weight, 0.0), budget))
        turnover = abs(weight - previous)
        nav *= 1.0 + weight * forward_return - turnover * TRANSACTION_COST
        weights.append(weight)
        previous = weight
    return np.asarray(weights, dtype=float)


def summarize_branch(branch: str, grid: pd.DataFrame) -> tuple[list[dict[str, object]], dict[str, object]]:
    grid = grid.copy()
    grid['beats_goal_baseline'] = grid['return_pct'] > baseline.return_pct
    grid['passes_drawdown_gate'] = grid['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR
    passes_both = grid.loc[grid['beats_goal_baseline'] & grid['passes_drawdown_gate']]
    best_return = grid.sort_values('return_pct', ascending=False).iloc[0]
    drawdown_pool = grid.loc[grid['passes_drawdown_gate']]
    best_drawdown = drawdown_pool.sort_values('return_pct', ascending=False).iloc[0] if not drawdown_pool.empty else grid.sort_values('max_drawdown_pct', ascending=False).iloc[0]
    rows = []
    for label_name, row in [('best_return', best_return), ('best_drawdown_gate', best_drawdown)]:
        decision = 'Promote' if bool(row['beats_goal_baseline']) and bool(row['passes_drawdown_gate']) else ('Drop' if not bool(row['passes_drawdown_gate']) else 'Iterate')
        decision_basis = 'beats_goal_baseline_and_drawdown_gate' if decision == 'Promote' else ('drawdown_gate' if not bool(row['passes_drawdown_gate']) else 'return_gate')
        rows.append(
            {
                'branch': branch,
                'candidate': f'{branch}_{label_name}',
                'coverage_start': START,
                'coverage_end': END,
                'return_pct': float(row['return_pct']),
                'max_drawdown_pct': float(row['max_drawdown_pct']),
                'baseline_return_pct': float(baseline.return_pct),
                'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
                'beats_goal_baseline': bool(row['beats_goal_baseline']),
                'passes_drawdown_gate': bool(row['passes_drawdown_gate']),
                'decision': decision,
                'decision_basis': decision_basis,
                'params': json.dumps(
                    {
                        key: (None if pd.isna(row[key]) else row[key].item() if hasattr(row[key], 'item') else row[key])
                        for key in grid.columns
                        if key not in {'return_pct', 'max_drawdown_pct', 'turnover', 'mean_exposure_pct', 'beats_goal_baseline', 'passes_drawdown_gate', 'raw_key'}
                    },
                    sort_keys=True,
                ),
            }
        )
    summary = {
        'coverage_start': START,
        'coverage_end': END,
        'grid_rows': int(len(grid)),
        'passes_both_count': int(len(passes_both)),
        'goal_baseline_return_pct': float(baseline.return_pct),
        'best_return_pct': float(best_return['return_pct']),
        'best_return_max_drawdown_pct': float(best_return['max_drawdown_pct']),
        'best_drawdown_gate_return_pct': float(best_drawdown['return_pct']),
        'best_drawdown_gate_max_drawdown_pct': float(best_drawdown['max_drawdown_pct']),
    }
    return rows, summary

hard_rows: list[dict[str, object]] = []
raw_candidates: dict[str, np.ndarray] = {}
for score_quantile in [0.0, 0.1, 0.2, 0.4, 0.55, 0.6, 0.65]:
    threshold = float(np.nanquantile(score, score_quantile))
    signal = (score > threshold).astype(float)
    for trend_name, trend in trend_flags.items():
        trend_signal = signal * trend.astype(float)
        for cap in [0.5, 0.75, 1.0, 1.25, 2.0, 2.5]:
            for vol_name, vol_values in vol_inputs.items():
                target_vols = [None] if vol_values is None else [0.5, 1.0, 1.5]
                for target_vol in target_vols:
                    raw = trend_signal * cap
                    if vol_values is not None:
                        raw = raw * np.clip(float(target_vol) / vol_values, 0.0, 2.0)
                    for vix_max in [None, 32, 45, 60]:
                        weight = raw if vix_max is None else np.where(vix <= vix_max, raw, 0.0)
                        metrics = path_metrics(weight)
                        raw_key = f'{score_quantile}|{trend_name}|{cap}|{vol_name}|{target_vol}|{vix_max}'
                        raw_candidates[raw_key] = weight.astype(float)
                        hard_rows.append(
                            {
                                'raw_key': raw_key,
                                'score_quantile': score_quantile,
                                'trend': trend_name,
                                'cap': cap,
                                'vol_name': vol_name,
                                'target_vol': target_vol,
                                'vix_max': vix_max,
                                **metrics,
                            }
                        )
hard_grid = pd.DataFrame(hard_rows)
hard_selected = pd.concat(
    [
        hard_grid.sort_values('return_pct', ascending=False).head(80),
        hard_grid.loc[hard_grid['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR].sort_values('return_pct', ascending=False).head(80),
        hard_grid.loc[hard_grid['max_drawdown_pct'] > -40].sort_values('return_pct', ascending=False).head(80),
    ]
).drop_duplicates('raw_key')

tipp_rows: list[dict[str, object]] = []
for _, base in hard_selected.iterrows():
    raw = raw_candidates[str(base['raw_key'])]
    for floor_dd in [0.12, 0.18, 0.2, 0.25, 0.3, 0.35]:
        for multiplier in [3, 5, 8, 12, 20]:
            for max_budget in [0.75, 1.0, 1.5, 2.0, 3.0]:
                weight = tipp_weight(raw, floor_dd=floor_dd, multiplier=multiplier, max_budget=max_budget)
                metrics = path_metrics(weight)
                tipp_rows.append(
                    {
                        'source_raw_key': base['raw_key'],
                        'score_quantile': base['score_quantile'],
                        'trend': base['trend'],
                        'cap': base['cap'],
                        'vol_name': base['vol_name'],
                        'target_vol': base['target_vol'],
                        'vix_max': base['vix_max'],
                        'floor_dd': floor_dd,
                        'multiplier': multiplier,
                        'max_budget': max_budget,
                        **metrics,
                    }
                )
tipp_grid = pd.DataFrame(tipp_rows)
hard_candidates, hard_summary = summarize_branch('production_signal_budget_overlay', hard_grid)
tipp_candidates, tipp_summary = summarize_branch('tipp_budget_overlay', tipp_grid)

candidate_metrics = pd.DataFrame(hard_candidates + tipp_candidates)
candidate_metrics.to_csv(CARD / 'agentic_alpha_budget_overlay_candidate_metrics.csv', index=False)
branch_results = {
    'production_signal_budget_overlay': hard_summary,
    'tipp_budget_overlay': tipp_summary,
}
passes_both_count = int(sum(result['passes_both_count'] for result in branch_results.values()))
best_candidate = None
if passes_both_count:
    promoted = candidate_metrics.loc[(candidate_metrics['return_pct'] > baseline.return_pct) & (candidate_metrics['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR)]
    best_candidate = str(promoted.sort_values('return_pct', ascending=False).iloc[0]['candidate'])

search_summary = {
    'workflow': 'agentic_alpha_budget_overlay_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'goal_achieved': bool(best_candidate),
    'best_candidate': best_candidate,
    'passes_both_count': passes_both_count,
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'branch_results': branch_results,
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'grid_counts': {
        'hard_gate_rows': int(len(hard_grid)),
        'tipp_budget_rows': int(len(tipp_grid)),
        'tipp_source_raw_candidates': int(len(hard_selected)),
    },
}
(CARD / 'agentic_alpha_budget_overlay_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'failure_modes.md').write_text(
    '# Failure Modes\n\n'
    '- production_signal_budget_overlay: high-return variants beat the production return only by accepting deep drawdowns; drawdown-gated variants retain too little exposure to beat the production baseline.\n'
    '- tipp_budget_overlay: CPPI/TIPP-style cushion budgeting bounds path drawdown but caps exposure during exactly the high-compounding periods needed to exceed the current production strategy.\n'
    '- next_action: test a new predictive model family or older intraday history; another budget overlay on the same production score is unlikely to close the full gate.\n',
    encoding='utf-8',
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        hard_gate_rows=len(hard_grid),
        tipp_budget_rows=len(tipp_grid),
        production_recorder_id=baseline.recorder_id,
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_budget_overlay_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
    for path in [
        CARD / 'agentic_alpha_budget_overlay_candidate_metrics.csv',
        CARD / 'agentic_alpha_budget_overlay_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_budget_overlay')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
(CARD / 'agentic_alpha_budget_overlay_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'workflow_config': 'workflow.yaml',
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'as_of': END,
    'data_range': [START, END],
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'branch_results': branch_results,
    'search_report': search_summary,
    'external_source_counts': search_summary['external_source_counts'],
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start', 'production_recorder.pred.pkl', 'production_recorder.label.pkl'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_budget_overlay_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'search_summary': 'agentic_alpha_budget_overlay_search_summary.json',
    },
}
summary['artifact_policy'] = {
    'policy': 'slim_card_recorder_first',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'workflow_config': 'workflow.yaml',
    'kept_sidecars': sorted(summary['sidecars'].values()),
    'reproducible_from_recorder': True,
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'signals': 'artifacts/signals.pkl',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')

conclusion = f'''# 中文结论

- 做了什么：继续 R-G-E-V-D，把 Github/论文/Arxiv/Twitter 的 LETF 趋势、vol-target、CPPI/TIPP 风险预算思路叠到现有 production recorder 的 `pred.pkl` 和 `label.pkl`，并通过 qlib `D.features` 读 SOXL/SOXX/VIX 状态后跑 hard gate 与 TIPP budget 两个分支。
- 核心指标：生产基准 {baseline.name} full-test return_pct={baseline.return_pct:.4f}、max_drawdown_pct={baseline.max_drawdown_pct:.4f}；本卡 passes_both_count={passes_both_count}。
- 分支结果：production_signal_budget_overlay 最高收益候选 return_pct={hard_summary['best_return_pct']:.4f}、max_drawdown_pct={hard_summary['best_return_max_drawdown_pct']:.4f}，最佳 20% 回撤候选 return_pct={hard_summary['best_drawdown_gate_return_pct']:.4f}、max_drawdown_pct={hard_summary['best_drawdown_gate_max_drawdown_pct']:.4f}；tipp_budget_overlay 最佳 20% 回撤候选 return_pct={tipp_summary['best_drawdown_gate_return_pct']:.4f}、max_drawdown_pct={tipp_summary['best_drawdown_gate_max_drawdown_pct']:.4f}。
- regime/window 差异：高收益路径集中在允许高暴露的趋势窗口，但这些路径在 2020/2022/2025-2026 的 SOXL 压力段出现深回撤；CPPI/TIPP 能压住回撤，但会把均值暴露压低到无法超过 production baseline。
- 是否晋升：不晋升；未找到同时满足收益超过现有生产策略且 MaxDD < 20% 的候选。
- 风险：这是 diagnostic search，不是生产门；它关闭的是“同一 production score 上继续叠 budget overlay”的分支，下一步应换预测模型族、补更长 intraday 历史，或研究明确的 crash-risk alpha。
- qlib recorder：{recorder_id}
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)
print('candidate_metrics=' + (CARD / 'agentic_alpha_budget_overlay_candidate_metrics.csv').name)
print('summary=' + (CARD / 'summary.json').name)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[19952:MainThread](2026-06-23 16:24:26,844) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[19952:MainThread](2026-06-23 16:24:26,846) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[19952:MainThread](2026-06-23 16:24:26,846) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[19952:MainThread](2026-06-23 16:25:00,360) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_budget_overlay_search_v1.


[19952:MainThread](2026-06-23 16:25:00,362) INFO - qlib.workflow - [exp.py:258] - Experiment 448614777724146373 starts running ...


[19952:MainThread](2026-06-23 16:25:00,434) INFO - qlib.workflow - [recorder.py:345] - Recorder d0d0260966724a83b4dbcc9a92f1c921 starts running under Experiment 448614777724146373 ...


[19952:MainThread](2026-06-23 16:25:00,522) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | waiting `async_log` Done


# 中文结论

- 做了什么：继续 R-G-E-V-D，把 Github/论文/Arxiv/Twitter 的 LETF 趋势、vol-target、CPPI/TIPP 风险预算思路叠到现有 production recorder 的 `pred.pkl` 和 `label.pkl`，并通过 qlib `D.features` 读 SOXL/SOXX/VIX 状态后跑 hard gate 与 TIPP budget 两个分支。
- 核心指标：生产基准 qlib_paper_sim_v1 full-test return_pct=6570.0233、max_drawdown_pct=-85.2317；本卡 passes_both_count=0。
- 分支结果：production_signal_budget_overlay 最高收益候选 return_pct=16107.3898、max_drawdown_pct=-94.8658，最佳 20% 回撤候选 return_pct=310.8535、max_drawdown_pct=-19.3817；tipp_budget_overlay 最佳 20% 回撤候选 return_pct=269.0667、max_drawdown_pct=-18.8160。
- regime/window 差异：高收益路径集中在允许高暴露的趋势窗口，但这些路径在 2020/2022/2025-2026 的 SOXL 压力段出现深回撤；CPPI/TIPP 能压住回撤，但会把均值暴露压低到无法超过 production baseline。
- 是否晋升：不晋升；未找到同时满足收益超过现有生产策略且 MaxDD < 20% 的候选。
- 风险：这是 diagnostic search，不是生产门；它关闭的是“同一 production score 上继续叠 budget overlay”的分支，下一步应换预测模型族、补更长 intraday 历史，或研究明确的 crash-risk alpha。
- qlib recorder：d0d0260966724a83b4dbcc9a92f1c921

candidate_metrics=agentic_alpha_budget_overlay_candidate_metrics.csv
summary=